In [1]:
import Pkg
Pkg.activate("../chebcoefs")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/chebcoefs`


In [2]:
using WaveGreen2D.Chebyshev: ChebyshevSeries, TransformedChebyshevSeries, ChebyshevCluster,
    gradient, hessian, normalize, contains, clenshaw, gradient_clenshaw, hessian_clenshaw,
    order, domain

In [3]:
using StaticArrays
using FastChebInterp
using Test

# include("Chebyshev.jl")
# using .Chebyshev
# import .Chebyshev: normalize, contains, clenshaw, gradient_clenshaw, hessian_clenshaw, order, domain

In [4]:
cs1 = ChebyshevSeries(zeros(2, 3), SA[9.10, -4.50], SA[1.30, 10.0])
cs2 = ChebyshevSeries(zeros(4, 5), SA[-1.9, -5.10], SA[1.35, 3.1])
cc = ChebyshevCluster(cs1, cs2)

Cluster of 2 2-D Chebyshev series: 
1. Order (1, 2), x ∈ [9.1, 1.3]×[-4.5, 10.0]
2. Order (3, 4), x ∈ [-1.9, 1.35]×[-5.1, 3.1]

In [5]:
@code_warntype ChebyshevCluster(cs1, cs2)

MethodInstance for ChebyshevCluster(::ChebyshevSeries{Float64, 2}, ::ChebyshevSeries{Float64, 2})
  from ChebyshevCluster(series::ChebyshevSeries{T, N}...) where {T, N} @ WaveGreen2D.Chebyshev ~/repos/WaveGreen2D/src/Chebyshev/Chebyshev.jl:153
Static Parameters
  T = Float64
  N = 2
Arguments
  #self#::Type{ChebyshevCluster}
  series::Tuple{ChebyshevSeries{Float64, 2}, ChebyshevSeries{Float64, 2}}
Locals
  M::Int64
Body::ChebyshevCluster{Float64, 2, 2}
1 ─ %1 = WaveGreen2D.Chebyshev.length::Core.Const(length)
│        (M = (%1)(series))
│   %3 = WaveGreen2D.Chebyshev.ChebyshevCluster::Core.Const(ChebyshevCluster)
│   %4 = $(Expr(:static_parameter, 1))::Core.Const(Float64)
│   %5 = $(Expr(:static_parameter, 2))::Core.Const(2)
│   %6 = M::Core.Const(2)
│   %7 = Core.apply_type(%3, %4, %5, %6)::Core.Const(ChebyshevCluster{Float64, 2, 2})
│   %8 = (%7)(series)::ChebyshevCluster{Float64, 2, 2}
└──      return %8



In [ ]:
fₘ₋₁

In [4]:
@testset "Series order" begin
    cs1 = ChebyshevSeries(zeros(10), -8.0, 4.0)
    cs2 = ChebyshevSeries(zeros(9, 5), SA[-2.0, -1.0], SA[1.0, 2.0])
    cs3 = ChebyshevSeries(zeros(4, 5, 3), SA[-5.0, -3.0, -0.5], SA[1.5, 2.0, 3.5])
    ts = TransformedChebyshevSeries(cs3)
    
    @test order(cs1) == 9
    @test order(cs2) == (8, 4)
    @test order(cs3) == (3, 4, 2)
    @test order(ts) == (3, 4, 2)
end

Test Summary: | Pass  Total  Time
Series order  |    4      4  0.4s


Test.DefaultTestSet("Series order", Any[], 4, false, false, true, 1.782017969345036e9, 1.782017969743305e9, false, "In[4]", Random.Xoshiro(0x308f3c42c34ab62c, 0x20c29914ca6dd1a0, 0x96bbd8745a7b70a9, 0xfbd363442e744c50, 0x6749f8a42c2b7489))

In [5]:
@testset "Series domain" begin
    cs1 = ChebyshevSeries(zeros(10), -8.0, 4.5989)
    cs2 = ChebyshevSeries(zeros(9, 5), SA[-2.1694, -1.3315], SA[1.0, 2.1125])
    cs3 = ChebyshevSeries(zeros(4, 5, 3), SA[-5.9994, -3.0, -0.5995], SA[1.5, 2.0, 3.5])
    ts = TransformedChebyshevSeries(cs3)
    
    @test domain(cs1) == "[-8.0, 4.599]"
    @test domain(cs2) == "[-2.169, 1.0]×[-1.332, 2.112]"
    @test domain(cs3) == "[-5.999, 1.5]×[-3.0, 2.0]×[-0.6, 3.5]"
    @test domain(ts) == "[-5.999, 1.5]×[-3.0, 2.0]×[-0.6, 3.5]"
end

Test Summary: | Pass  Total  Time
Series domain |    4      4  0.1s


Test.DefaultTestSet("Series domain", Any[], 4, false, false, true, 1.782017970227752e9, 1.78201797033579e9, false, "In[5]", Random.Xoshiro(0x308f3c42c34ab62c, 0x20c29914ca6dd1a0, 0x96bbd8745a7b70a9, 0xfbd363442e744c50, 0x6749f8a42c2b7489))

In [6]:
@testset "Series show" begin
    io = IOBuffer()
    
    cs1 = ChebyshevSeries(zeros(17), 10.0, 90.0)
    cs2 = ChebyshevSeries(zeros(7, 9), SA[-1.900, -8.6667], SA[2.2225, 1.00])
    cs3 = ChebyshevSeries(zeros(5, 10, 3), SA[-3.51, 0.343, -3.729], SA[4.56, 6.258, 4.96])
    
    cs1_long = sprint((io, x) -> show(io, MIME"text/plain"(), x), cs1)
    cs1_short = sprint((io, x) -> show(io, x), cs1)

    cs2_long = sprint((io, x) -> show(io, MIME"text/plain"(), x), cs2)
    cs2_short = sprint((io, x) -> show(io, x), cs2)

    cs3_long = sprint((io, x) -> show(io, MIME"text/plain"(), x), cs3)
    cs3_short = sprint((io, x) -> show(io, x), cs3)

    @test cs1_short == "1-D Chebyshev series of order 16"
    @test cs1_long == "1-dimensional Chebyshev series of order 16 for x ∈ [10.0, 90.0]"

    @test cs2_short == "2-D Chebyshev series of order (6, 8)"
    @test cs2_long == "2-dimensional Chebyshev series of order (6, 8) for x ∈ [-1.9, 2.222]×[-8.667, 1.0]"

    @test cs3_short == "3-D Chebyshev series of order (4, 9, 2)"
    @test cs3_long == "3-dimensional Chebyshev series of order (4, 9, 2) for x ∈ [-3.51, 4.56]×[0.343, 6.258]×[-3.729, 4.96]"
end

Test Summary: | Pass  Total  Time
Series show   |    6      6  0.2s


Test.DefaultTestSet("Series show", Any[], 6, false, false, true, 1.782017971214616e9, 1.782017971445824e9, false, "In[6]", Random.Xoshiro(0x308f3c42c34ab62c, 0x20c29914ca6dd1a0, 0x96bbd8745a7b70a9, 0xfbd363442e744c50, 0x6749f8a42c2b7489))

In [7]:
@testset "Transformed series show" begin
    io = IOBuffer()
    
    cs1 = ChebyshevSeries(zeros(13), 11.1115, 90.6669)
    cs2 = ChebyshevSeries(zeros(5, 7), SA[-1.91, -8.60], SA[1.25, 3.0])
    cs3 = ChebyshevSeries(zeros(2, 4, 5), SA[-1.42, -2.125, 0.7542], SA[5.46, 3.278, 2.86])
    
    ts1 = TransformedChebyshevSeries(cs1)
    ts2 = TransformedChebyshevSeries(cs2)
    ts3 = TransformedChebyshevSeries(cs3)
    
    ts1_long = sprint((io, x) -> show(io, MIME"text/plain"(), x), ts1)
    ts1_short = sprint((io, x) -> show(io, x), ts1)

    ts2_long = sprint((io, x) -> show(io, MIME"text/plain"(), x), ts2)
    ts2_short = sprint((io, x) -> show(io, x), ts2)

    ts3_long = sprint((io, x) -> show(io, MIME"text/plain"(), x), ts3)
    ts3_short = sprint((io, x) -> show(io, x), ts3)

    @test ts1_short == "1-D transformed Chebyshev series of order 12"
    @test ts1_long == "1-dimensional transformed Chebyshev series of order 12 for u(x) ∈ [11.112, 90.667]"

    @test ts2_short == "2-D transformed Chebyshev series of order (4, 6)"
    @test ts2_long == "2-dimensional transformed Chebyshev series of order (4, 6) for u(x) ∈ [-1.91, 1.25]×[-8.6, 3.0]"

    @test ts3_short == "3-D transformed Chebyshev series of order (1, 3, 4)"
    @test ts3_long == "3-dimensional transformed Chebyshev series of order (1, 3, 4) for u(x) ∈ [-1.42, 5.46]×[-2.125, 3.278]×[0.754, 2.86]"
end

Test Summary:           | Pass  Total  Time
Transformed series show |    6      6  0.3s


Test.DefaultTestSet("Transformed series show", Any[], 6, false, false, true, 1.782017971591717e9, 1.782017971872436e9, false, "In[7]", Random.Xoshiro(0x308f3c42c34ab62c, 0x20c29914ca6dd1a0, 0x96bbd8745a7b70a9, 0xfbd363442e744c50, 0x6749f8a42c2b7489))

In [8]:
@testset "Cluster show" begin
    io = IOBuffer()
    
    cs1 = ChebyshevSeries(zeros(2, 3), SA[9.10, -4.50], SA[1.30, 10.0])
    cs2 = ChebyshevSeries(zeros(4, 5), SA[-1.9, -5.10], SA[1.35, 3.1])
    cc = ChebyshevCluster(cs1, cs2)
    
    cc_long = sprint((io, x) -> show(io, MIME"text/plain"(), x), cc)
    cc_short = sprint((io, x) -> show(io, x), cc)

    @test cc_short == "Cluster of 2 2-D Chebyshev series"
    @test cc_long == "Cluster of 2 2-D Chebyshev series: \n1. Order (1, 2), x ∈ [9.1, 1.3]×[-4.5, 10.0]\n2. Order (3, 4), x ∈ [-1.9, 1.35]×[-5.1, 3.1]"
end

Test Summary: | Pass  Total  Time
Cluster show  |    2      2  0.3s


Test.DefaultTestSet("Cluster show", Any[], 2, false, false, true, 1.782017972369139e9, 1.782017972665607e9, false, "In[8]", Random.Xoshiro(0x308f3c42c34ab62c, 0x20c29914ca6dd1a0, 0x96bbd8745a7b70a9, 0xfbd363442e744c50, 0x6749f8a42c2b7489))

In [14]:
f(x) = sin(x)
lb, ub = SA[0.0], SA[0.5π]
xc = chebpoints(50, lb[], ub[])
cb = chebinterp(f.(xc), lb[], ub[]) 
cs = ChebyshevSeries(cb.coefs, lb, ub)
print(cs)
cs

1-D Chebyshev series of order 13

1-dimensional Chebyshev series of order 13 for x ∈ [0.0, 1.571]

In [15]:
@testset "1-D series show" begin
    cs = ChebyshevSeries(zeros(2), -5.5549, 8.4854)
end

Test Summary:   | Total  Time
1-D series show |     0  0.0s


Test.DefaultTestSet("1-D series show", Any[], 0, false, false, true, 1.782010985214819e9, 1.782010985214849e9, false, "In[15]", Random.Xoshiro(0x113a30ba047f98a9, 0x71860398ec0a3910, 0xdb4e4e302988e52b, 0xd59d9d17484796a0, 0x85a30e18923fdc10))

In [16]:
f(x) = cos(0.25*x[1]*x[2])
lb = SA[-0.05, 0.2]
ub = SA[0.15, 0.4]
xc = chebpoints((50, 50), lb, ub)
cb = chebinterp(f.(xc), lb, ub)
cs = ChebyshevSeries(cb.coefs, lb, ub)
print(cs)
cs

2-D Chebyshev series of order (4, 4)

2-dimensional Chebyshev series of order (4, 4) for x ∈ [-0.05, 0.15]×[0.2, 0.4]

In [5]:
f(x) = exp(x[1]*x[2]) * cos(x[1] + x[3]/2)
lb = SA[0.5, -0.2, 1.0]
ub = SA[0.7, 0.0, 1.2]
xc = chebpoints((50, 50, 50), lb, ub)
cb = chebinterp(f.(xc), lb, ub)
cs = ChebyshevSeries(cb.coefs, lb, ub)
print(cs)
cs

3-D Chebyshev series of order (8, 7, 7)

3-dimensional Chebyshev series of order (8, 7, 7) for x ∈ [0.500, 0.700]×[-0.200, 0.000]×[1.000, 1.200]

In [6]:
f(u) = sin(u^2)
u(x::SVector{1,Float64}) = x .^ 0.5
∇u(x::SVector{1,Float64}) = reshape(0.5 * x .^ -0.5, Size(1, 1))
Hu(x::SVector{1,Float64}) = reshape(-0.25 * x .^ -1.5, Size(1, 1, 1))

lb, ub = SA[0.0], SA[1.5]
uc = chebpoints(50, lb[], ub[])
cb = chebinterp(f.(uc), lb[], ub[]) 
cs = ChebyshevSeries(cb.coefs, lb, ub)
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu)
print(ts)
ts

1-D transformed Chebyshev series of order 22

1-dimensional transformed Chebyshev series of order 22 for u(x) ∈ [0.000, 1.500]

In [7]:
function f(u::SVector{2, Float64})
    r, θ = u
    return exp(r*cos(θ)) * cos(r*sin(θ))
end

function u(x::SVector{2,Float64})
    ξ, η = x

    r = sqrt(ξ^2 + η^2)
    θ = atan(η / ξ)

    return SA[r, θ]
end

function ∇u(x::SVector{2,Float64})
    ξ, η = x

    r² = ξ^2 + η^2
    r = √r²

    r₁ = ξ / r
    r₂ = η / r
    θ₁ = -η / r²
    θ₂ = ξ / r²

    ∇u₁ = SA[r₁, r₂]
    ∇u₂ = SA[θ₁, θ₂]

    return vcat(∇u₁', ∇u₂')
end

function Hu(x::SVector{2,Float64})
    ξ, η = x

    ξ², η² = ξ^2, η^2

    r² = ξ² + η²
    r = √r²
    r³ = r² * r
    r⁴ = r³ * r

    r₁₁ = η² / r³
    r₁₂ = -ξ * η / r³
    r₂₁ = r₁₂
    r₂₂ = ξ² / r³

    θ₁₁ = 2 * ξ * η / r⁴
    θ₁₂ = (η² - ξ²) / r⁴
    θ₂₁ = θ₁₂
    θ₂₂ = -2 * ξ * η / r⁴

    Hu₁ = reshape([r₁₁ r₁₂;
            r₂₁ r₂₂], 1, 2, 2)

    Hu₂ = reshape([θ₁₁ θ₁₂;
            θ₂₁ θ₂₂], 1, 2, 2)

    hess_u = vcat(Hu₁, Hu₂)

    return SArray{Tuple{2,2,2},Float64}(hess_u)
end

lb = SA[0.1, -0.7]
ub = SA[2.0, 1.3]
uc = chebpoints((50, 50), lb, ub)
cb = chebinterp(f.(uc), lb, ub)
cs = ChebyshevSeries(cb.coefs, lb, ub)
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu)
print(ts)
ts

2-D transformed Chebyshev series of order (14, 31)

2-dimensional transformed Chebyshev series of order (14, 31) for u(x) ∈ [0.100, 2.000]×[-0.700, 1.300]

In [8]:
function f(u::SVector{3, Float64})
    r, θ, ϕ = u
    r² = r^2
    return r² * sin(ϕ) * cos(θ) * cos(ϕ) * exp(-r²)
end

function u(x::SVector{3,Float64})
    ξ, η, ζ = x

    ρ = sqrt(ξ^2 + η^2)

    r = sqrt(ξ^2 + η^2 + ζ^2)
    θ = atan(η / ξ)
    ϕ = atan(ρ / ζ)

    return SA[r, θ, ϕ]
end

function ∇u(x::SVector{3,Float64})
    ξ, η, ζ = x

    r² = ξ^2 + η^2 + ζ^2
    r = √r²
    ρ² = ξ^2 + η^2
    ρ = √ρ²

    r₁ = ξ / r
    r₂ = η / r
    r₃ = ζ / r
    θ₁ = -η / ρ²
    θ₂ = ξ / ρ²
    θ₃ = 0
    ϕ₁ = ξ * ζ / (ρ * r²)
    ϕ₂ = η * ζ / (ρ * r²)
    ϕ₃ = -ρ / r²

    ∇u₁ = SA[r₁, r₂, r₃]
    ∇u₂ = SA[θ₁, θ₂, θ₃]
    ∇u₃ = SA[ϕ₁, ϕ₂, ϕ₃]

    return vcat(∇u₁', ∇u₂', ∇u₃')
end

function Hu(x::SVector{3,Float64})
    ξ, η, ζ = x

    ξ² = ξ^2
    η² = η^2
    ζ² = ζ^2

    r² = ξ² + η² + ζ²
    r = √r²
    r³ = r² * r
    r⁴ = r³ * r

    ρ² = ξ² + η²
    ρ = √ρ²
    ρ³ = ρ² * ρ
    ρ⁴ = ρ³ * ρ

    r₁₁ = (η² + ζ²) / r³
    r₁₂ = -ξ * η / r³
    r₁₃ = -ξ * ζ / r³
    r₂₁ = r₁₂
    r₂₂ = (ξ² + ζ²) / r³
    r₂₃ = -η * ζ / r³
    r₃₁ = r₁₃
    r₃₂ = r₂₃
    r₃₃ = ρ² / r³

    θ₁₁ = 2 * ξ * η / ρ⁴
    θ₁₂ = (η² - ξ²) / ρ⁴
    θ₁₃ = 0
    θ₂₁ = θ₁₂
    θ₂₂ = -θ₁₁
    θ₂₃ = 0
    θ₃₁ = θ₁₃
    θ₃₂ = θ₂₃
    θ₃₃ = 0

    ϕ₁₁ = ζ * (-ξ² * ζ² - 3 * ξ² * ρ² + ρ² * r²) / (ρ³ * r⁴)
    ϕ₁₂ = -ξ * η * ζ * (3 * ξ² + 3 * η² + ζ²) / (ρ³ * r⁴)
    ϕ₁₃ = ξ * (ρ² - ζ²) / (ρ * r⁴)
    ϕ₂₁ = ϕ₁₂
    ϕ₂₂ = ζ * (-η² * ζ² - 3 * η² * ρ² + ρ² * r²) / (ρ³ * r⁴)
    ϕ₂₃ = η * (ρ² - ζ²) / (ρ * r⁴)
    ϕ₃₁ = ϕ₁₃
    ϕ₃₂ = ϕ₂₃
    ϕ₃₃ = 2ζ * ρ / r⁴

    Hu₁ = reshape([r₁₁ r₁₂ r₁₃;
            r₂₁ r₂₂ r₂₃;
            r₃₁ r₃₂ r₃₃], 1, 3, 3)

    Hu₂ = reshape([θ₁₁ θ₁₂ θ₁₃;
            θ₂₁ θ₂₂ θ₂₃;
            θ₃₁ θ₃₂ θ₃₃], 1, 3, 3)

    Hu₃ = reshape([ϕ₁₁ ϕ₁₂ ϕ₁₃;
            ϕ₂₁ ϕ₂₂ ϕ₂₃;
            ϕ₃₁ ϕ₃₂ ϕ₃₃], 1, 3, 3)

    hess_u = vcat(Hu₁, Hu₂, Hu₃)

    return SArray{Tuple{3,3,3},Float64}(hess_u)
end

lb = SA[0.1, 0.2, 0.3]
ub = SA[2.2, 1.8, 1.6]
uc = chebpoints((50, 50, 50), lb, ub)
cb = chebinterp(f.(uc), lb, ub)
cs = ChebyshevSeries(cb.coefs, lb, ub)
ts = TransformedChebyshevSeries(cs, u, ∇u, Hu)
print(ts)
ts

3-D transformed Chebyshev series of order (26, 13, 16)

3-dimensional transformed Chebyshev series of order (26, 13, 16) for u(x) ∈ [0.100, 2.200]×[0.200, 1.800]×[0.300, 1.600]

In [9]:
f(x) = exp(cos(0.5x))

lb1, ub1 = SA[0.0], SA[0.5]
lb2, ub2 = SA[0.5], SA[1.0]

xc1 = chebpoints(50, lb1[], ub1[])
cb1 = chebinterp(f.(xc1), lb1[], ub1[])
cs1 = ChebyshevSeries(cb1.coefs, lb1, ub1)

xc2 = chebpoints(50, lb2[], ub2[])
cb2 = chebinterp(f.(xc2), lb2[], ub2[])
cs2 = ChebyshevSeries(cb2.coefs, lb2, ub2)

cc = ChebyshevCluster(cs1, cs2)
print(cc)
cc

Cluster of 2 1-D Chebyshev series

Cluster of 2 1-D Chebyshev series: 
1. Order 10, x ∈ [0.000, 0.500]
2. Order 10, x ∈ [0.500, 1.000]

In [10]:
cs1 = ChebyshevSeries(zeros(2, 2), SA[0.0, 1.0], SA[1.0, 2.0])
cs2 = ChebyshevSeries(zeros(2, 2), SA[1.0, 2.0], SA[2.0, 3.0])
cc = ChebyshevCluster(cs1, cs2)

print(cc)
cc

Cluster of 2 2-D Chebyshev series

Cluster of 2 2-D Chebyshev series: 
1. Order (1, 1), x ∈ [0.000, 1.000]×[1.000, 2.000]
2. Order (1, 1), x ∈ [1.000, 2.000]×[2.000, 3.000]